# Local Green, Global Red III: 01 Two Passes, One Failure
## How can both parts pass while the composite fails?

**Author:** Jeremy H. Carroll  
**Date:** 2026-10-15  
**Version:** 1.5.2  

---

### 1. Question and Prediction

> **Question:** Can two individually passing components produce an integrated failure?

**Prediction:** Before running the code below, consider: if Component 1 passes its unit test ($e_1 = 1$) and Component 2 passes its unit test ($e_2 = 1$), must the composed pipeline preserve correctness?

The answer is **no**: local tests verify behavior against local preconditions, but cannot guarantee that boundary assumptions between components are mutually compatible.

In [ ]:
from io import BytesIO
from IPython.display import Image, display
import matplotlib.pyplot as plt

def display_figure(fig):
    """Embed PNG bytes directly to ensure rendering under headless Agg backend."""
    try:
        with BytesIO() as buffer:
            fig.savefig(buffer, format='png', dpi=160, bbox_inches='tight')
            display(Image(data=buffer.getvalue()))
    finally:
        plt.close(fig)

# Elementary Defect Formula
def compute_defect(e1: int, e2: int, eg: int) -> int:
    if any(type(v) is not int or v not in (0, 1) for v in (e1, e2, eg)):
        raise ValueError('Verdicts must be integers in {0, 1}')
    return eg - (e1 * e2)

# Concrete Software Example: Producer and Consumer
# Producer generates a non-empty string identifier
# Consumer expects a digit-only numeric string
def producer(x: int) -> str:
    return f'ID_{x}'  # Passes local check: non-empty string

def consumer(s: str) -> int:
    return int(s)      # Passes local check on digit strings

# Local Unit Tests
e1 = 1 if len(producer(42)) > 0 else 0
e2 = 1 if consumer('42') == 42 else 0

# Integrated Composite Pipeline
try:
    res = consumer(producer(42))
    eg = 1 if res == 42 else 0
except Exception:
    eg = 0  # Failed integration

delta = compute_defect(e1, e2, eg)
print(f'Producer local verdict (e1):        {e1} (Pass)')
print(f'Consumer local verdict (e2):        {e2} (Pass)')
print(f'Integrated composite verdict (eg):  {eg} (Fail)')
print(f'Evaluator defect (Delta_E):         {delta} (LGGR witness)')

### 2. The Complete Truth Table: All Eight Triples

There are exactly $2^3 = 8$ possible combinations of $(e_1, e_2, e_g)$. Notice that zero defect $(\Delta_E = 0)$ means **agreement with the local product**, not that all components are green.

In [ ]:
triples = [(e1, e2, eg) for e1 in (0, 1) for e2 in (0, 1) for eg in (0, 1)]
table_data = []
for e1, e2, eg in triples:
    d = compute_defect(e1, e2, eg)
    prod = e1 * e2
    if d == -1:
        regime = 'Local Green, Global Red (LGGR)'
    elif d == 1:
        regime = 'Global-only pass'
    else:
        regime = 'Agreement with local product'
    table_data.append((e1, e2, prod, eg, d, regime))

print(f"{'e1':^4} | {'e2':^4} | {'e1*e2':^6} | {'eg':^4} | {'Delta_E':^8} | Regime")
print('-' * 60)
for e1, e2, prod, eg, d, reg in table_data:
    print(f"{e1:^4} | {e2:^4} | {prod:^6} | {eg:^4} | {d:^8} | {reg}")

# Visual Plot with explicit (0, 0, 0) markers
labels = [f'({e1}, {e2}, {eg})' for e1, e2, eg in triples]
defects = [compute_defect(e1, e2, eg) for e1, e2, eg in triples]
colors = ['#c44536' if d < 0 else '#2f6690' if d > 0 else '#888888' for d in defects]

fig, ax = plt.subplots(figsize=(9, 3.6))
bars = ax.bar(labels, defects, color=colors, width=0.5, edgecolor='#333333', linewidth=1)
ax.axhline(0, color='#222222', linewidth=1.0)
# Place visible markers on zero-defect cases so they are not invisible flat bars
for idx, d in enumerate(defects):
    if d == 0:
        ax.scatter(idx, 0, color='#888888', s=60, zorder=3, label='Agreement' if idx == 0 else '')
ax.set_ylim(-1.4, 1.4)
ax.set_yticks([-1, 0, 1])
ax.set_yticklabels(['-1 (LGGR)', '0 (Agreement)', '+1 (Global Only)'])
ax.set_xlabel('Verdict Triple (e1, e2, eg)', fontsize=10)
ax.set_ylabel('Signed Defect Delta_E', fontsize=10)
ax.set_title('Evaluator Defect Trichotomy Across All Eight Binary Triples', fontsize=11, pad=10)
ax.grid(axis='y', linestyle='--', alpha=0.4)
display_figure(fig)

### 3. Interactive Verdict Explorer (Optional Widget)

The cell below establishes a static baseline first, then optionally loads interactive controls if `ipywidgets` is present.

In [ ]:
import os

def evaluate_triple_ui(e1: int = 1, e2: int = 1, eg: int = 0):
    delta = compute_defect(e1, e2, eg)
    prod = e1 * e2
    if delta == -1:
        status = 'LOCAL GREEN, GLOBAL RED: Component tests pass, composition fails.'
    elif delta == 1:
        status = 'GLOBAL PASS ONLY: Composite passes while one or more components fail.'
    elif prod == 1 and eg == 1:
        status = 'COMPLETE SUCCESS: All local components and composition pass.'
    else:
        status = 'AGREEMENT AT FAILURE: Failure correctly propagated from components.'
    print(f'Selected Triple: (e1={e1}, e2={e2}, eg={eg})')
    print(f'Local Product (e1 * e2): {prod}')
    print(f'Evaluator Defect Delta_E: {delta}')
    print(f'Interpretation: {status}')

# Baseline static evaluation
print('--- Static Baseline Evaluation ---')
evaluate_triple_ui(1, 1, 0)

# Optional interactive widget
has_widgets = os.environ.get('LGGR3_HEADLESS') != '1'
if has_widgets:
    try:
        import ipywidgets as widgets
        from IPython.display import display
        w_e1 = widgets.Dropdown(options=[('Fail (0)', 0), ('Pass (1)', 1)], value=1, description='e1:')
        w_e2 = widgets.Dropdown(options=[('Fail (0)', 0), ('Pass (1)', 1)], value=1, description='e2:')
        w_eg = widgets.Dropdown(options=[('Fail (0)', 0), ('Pass (1)', 1)], value=0, description='eg:')
        ui = widgets.VBox([widgets.HBox([w_e1, w_e2, w_eg])])
        out = widgets.interactive_output(evaluate_triple_ui, {'e1': w_e1, 'e2': w_e2, 'eg': w_eg})
        display(ui, out)
    except ImportError:
        print('\n(ipywidgets not installed: interactive dropdowns disabled)')
else:
    print('\n(Headless execution: interactive widget skipped)')

### 4. Exercise, Worked Answer, and Evidence Boundary

**Exercise:**  
1. Find a triple with $\Delta_E = +1$.  
2. Explain why $\Delta_E = 0$ does not mean that the system is fully working.

---

**Worked Answer:**  
1. Triples $(0, 0, 1), (0, 1, 1), (1, 0, 1)$ all have $\Delta_E = +1$. In each case, $e_1 e_2 = 0$ but $e_g = 1$.  
2. The triple $(0, 0, 0)$ has $\Delta_E = 0 - 0 = 0$. Both components failed, and the composite failed. Zero defect measures agreement between the composite and the local product ($e_g = e_1 e_2$), NOT all-green success.

---

**Evidence Boundary:**  
This lesson proves the finite arithmetic trichotomy $\Delta_E \in \{-1, 0, 1\}$ for binary evaluations. It does not claim statistical reliability bounds or system safety guarantees.